# 산학프로젝트
**제조 공정 품질 불량 예측**

# 04_1_Imbalance_and_Feature_Experiments

## import

In [1]:
import pandas as pd
import numpy as np
import os

## 경로설정

In [2]:
os.chdir("data")

## 데이터 불러오기

In [3]:
labeled=pd.read_csv('labeled_data.csv', encoding='utf-8')
result_df=pd.read_csv('03_Baseline_Modeling_Result.csv')

In [4]:
labeled.head()

,_id,TimeStamp,PART_FACT_PLAN_DATE,PART_FACT_SERIAL,PART_NAME,EQUIP_CD,EQUIP_NAME,PassOrFail,Reason,Injection_Time,...,Mold_Temperature_3,Mold_Temperature_4,Mold_Temperature_5,Mold_Temperature_6,Mold_Temperature_7,Mold_Temperature_8,Mold_Temperature_9,Mold_Temperature_10,Mold_Temperature_11,Mold_Temperature_12
0,5f8928bb9c0189cc666ef19b,2020-10-16 04:57:47,2020-10-16 오전 12:00:00,24,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기,Y,NaN,9.59,...,24.799999,27.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5f8928de9c0189cc666ef20b,2020-10-16 04:58:48,2020-10-16 오전 12:00:00,24,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기,Y,NaN,9.60,...,24.799999,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5f8928df9c0189cc666ef213,2020-10-16 04:58:48,2020-10-16 오전 12:00:00,23,CN7 W/S SIDE MLD'G LH,S14,650톤-우진2호기,Y,NaN,9.60,...,24.799999,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5f8928f39c0189cc666ef25e,2020-10-16 04:59:48,2020-10-16 오전 12:00:00,23,CN7 W/S SIDE MLD'G LH,S14,650톤-우진2호기,Y,NaN,9.59,...,25.000000,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5f8928f59c0189cc666ef265,2020-10-16 04:59:48,2020-10-16 오전 12:00:00,24,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기,Y,NaN,9.59,...,25.000000,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
result_df

,Model,Defect Precision,Defect Recall,Defect F1 Score,Average Precision
0,RandomForest Baseline,0.294118,0.416667,0.344828,0.452946
1,Bagging Baseline,0.368421,0.583333,0.451613,0.444408
2,XGBoost Baseline,0.368421,0.583333,0.451613,0.269109


## 모델 개선 실험

In [7]:
# 개선 실험 공통 라이브러리
from pathlib import Path
import sys

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn import set_config
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import precision_recall_curve, average_precision_score, classification_report
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'data' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 공통 평가 및 튜닝 함수
from utils.model_evaluation import evaluate_models, show_random_forest_feature_importance
from utils.model_tuning import grid_search_random_forest_smote

set_config(display='text')

### 클래스 가중치

#### 데이터 전처리

In [ ]:
def info_df(df):
    info=pd.DataFrame(
        {
            "Columns": df.columns,
            "Non-Null Count": df.notna().sum().values,
            "Dtype": df.dtypes.values
        }
    )
    return info

In [ ]:
info_df(labeled)

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12', 'Mold_Temperature_2',
               'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL',
               'PART_NAME', 'Reason', 'Switch_Over_Position', 'TimeStamp', '_id']

final_df=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
final_df['PassOrFail']=final_df['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

final_df.shape

In [ ]:
final_df.head()

In [ ]:
# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(final_df.drop(columns='PassOrFail'), final_df['PassOrFail'], test_size=0.2, stratify=final_df['PassOrFail'], random_state=0)

print("데이터 분할 결과")
print(f'X_Train: {x_train.shape}, Y_Train: {y_train.shape}')
print(f'X_Test: {x_test.shape}, Y_Test: {y_test.shape}')

# 데이터 스케일링
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

#### 모델 학습

In [ ]:
model_dict={}
balanced_rf=RandomForestClassifier(n_estimators=1000, class_weight='balanced', min_samples_leaf=2, n_jobs=1, random_state=0)
balanced_rf.fit(x_train_scaled, y_train)
model_dict['RandomForest Balanced']=balanced_rf

#### 예측

In [ ]:
# model_evaluation.py
result_df=evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
importance_tables = show_random_forest_feature_importance(
    model_dict, x_train.columns
)

### SMOTE: Sampling

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12', 'Mold_Temperature_2',
               'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL',
               'PART_NAME', 'Reason', 'Switch_Over_Position', 'TimeStamp', '_id']

final_df=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
final_df['PassOrFail']=final_df['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 데이터 분할
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=y, random_state=0)

scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# 학습데이터에 SMOTE 적용
from imblearn.over_sampling import SMOTE

smote=SMOTE(random_state=0)
print("Before")
print(x_train_scaled.shape, y_train.shape)

x_train_smote, y_train_smote=smote.fit_resample(x_train_scaled, y_train)
print("After")
print(x_train_smote.shape, y_train_smote.shape)

#### 모델 학습

In [ ]:
# 학습 모델 저장
model_dict={} # evaluate_models가 for 문으로 작동하기 때문에 초기화 필요

# Random Forest
smote_rf=RandomForestClassifier(n_estimators=1000, random_state=0)
smote_rf.fit(x_train_smote, y_train_smote)
model_dict['RandomForest SMOTE']=smote_rf

# Bagging
smote_bag=BaggingClassifier(n_estimators=1000, random_state=0)
smote_bag.fit(x_train_smote, y_train_smote)
model_dict['Bagging SMOTE']=smote_bag

# XGBoost
smote_xgb=XGBClassifier(n_estimators=1000, random_state=0)
smote_xgb.fit(x_train_smote, y_train_smote)
model_dict['XGBoost SMOTE']=smote_xgb

#### 예측

In [ ]:
# model_dict의 키를 모델명으로 그대로 사용해 예측 및 평가
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
importance_tables=show_random_forest_feature_importance(model_dict, x_train.columns)

#### SMOTE 파라미터 조정

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12', 'Mold_Temperature_2',
               'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL',
               'PART_NAME', 'Reason', 'Switch_Over_Position', 'TimeStamp', '_id']

final_df=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
final_df['PassOrFail']=final_df['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 데이터 분할
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=y, random_state=0)

# 스케일링
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# SMOTE Sampling_strategy
smote=SMOTE(sampling_strategy=0.1, k_neighbors=3, random_state=0)
print("Before")
print(x_train_scaled.shape, y_train.shape)

x_train_smote, y_train_smote=smote.fit_resample(x_train_scaled, y_train)
print("After")
print(x_train_smote.shape, y_train_smote.shape)

#### 모델 학습

In [ ]:
# 학습 모델 저장
model_dict={}

# Random Forest
smote_rf=RandomForestClassifier(n_estimators=1000, random_state=0)
smote_rf.fit(x_train_smote, y_train_smote)
model_dict['RandomForest SMOTE St=0.1']=smote_rf

# Bagging
smote_bag=BaggingClassifier(n_estimators=1000, random_state=0)
smote_bag.fit(x_train_smote, y_train_smote)
model_dict['Bagging SMOTE St=0.1']=smote_bag

# XGBoost
smote_xgb=XGBClassifier(n_estimators=1000, random_state=0)
smote_xgb.fit(x_train_smote, y_train_smote)
model_dict['XGBoost SMOTE St=0.1']=smote_xgb

#### 예측

In [ ]:
# model_dict의 키를 모델명으로 그대로 사용해 예측 및 평가
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
importance_tables = show_random_forest_feature_importance(
    model_dict, x_train.columns
)

### 제품명 변수 추가

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# CN7, RG3 구분하는 변수 추가
drop_part['Part']=drop_part['PART_NAME'].str[:3]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11',
              'Mold_Temperature_12', 'Mold_Temperature_2','Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7',
              'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL','PART_NAME', 'Reason',
              'Switch_Over_Position', 'TimeStamp', '_id']

drop_col=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
drop_col['PassOrFail']=drop_col['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 더미변수 생성
final_df = pd.get_dummies(data = drop_col, columns = ['Part'], drop_first=True)
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=final_df['PassOrFail'], random_state=0)

# 스케일러
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

#### 모델 학습

In [ ]:
model_dict={}
# Random Forest
rf=RandomForestClassifier(n_estimators=1000, random_state=0)
rf.fit(x_train_scaled, y_train)
model_dict['RandomForest PART_NAME_1']=rf

#### 예측

In [ ]:
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# CN7, RG3 / LH, RH 구분하는 변수 추가
drop_part['Part']=drop_part['PART_NAME'].str[:3]+drop_part['PART_NAME'].str[-2:]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11',
              'Mold_Temperature_12', 'Mold_Temperature_2','Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7',
              'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL','PART_NAME', 'Reason',
              'Switch_Over_Position', 'TimeStamp', '_id']

drop_col=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
drop_col['PassOrFail']=drop_col['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 더미변수 생성
final_df = pd.get_dummies(data = drop_col, columns = ['Part'], drop_first=True)
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=final_df['PassOrFail'], random_state=0)

# 스케일러
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(final_df.drop(columns='PassOrFail'), final_df['PassOrFail'], test_size=0.2, stratify=final_df['PassOrFail'], random_state=0)

# 스케일러
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

#### 모델 학습

In [ ]:
model_dict={}
# Random Forest
rf=RandomForestClassifier(n_estimators=1000, random_state=0)
rf.fit(x_train_scaled, y_train)
model_dict['RandomForest PART_NAME_2']=rf

#### 예측

In [ ]:
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
importance_tables = show_random_forest_feature_importance(
    model_dict, x_train.columns
)

### PART_NAME + Class Weight

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# CN7, RG3 / LH, RH 구분하는 변수 추가
drop_part['Part']=drop_part['PART_NAME'].str[:3]+drop_part['PART_NAME'].str[-2:]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11',
              'Mold_Temperature_12', 'Mold_Temperature_2','Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7',
              'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL','PART_NAME', 'Reason',
              'Switch_Over_Position', 'TimeStamp', '_id']

drop_col=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
drop_col['PassOrFail']=drop_col['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 더미변수 생성
final_df = pd.get_dummies(data = drop_col, columns = ['Part'], drop_first=True)
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=final_df['PassOrFail'], random_state=0)

# 스케일러
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

#### 모델 학습

In [ ]:
model_dict={}
# Random Forest
rf=RandomForestClassifier(n_estimators=1000, class_weight='balanced', min_samples_leaf=2, n_jobs=1,random_state=0)
rf.fit(x_train_scaled, y_train)
model_dict['RandomForest PART_NAME + Balanced']=rf

#### 예측

In [ ]:
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
importance_tables = show_random_forest_feature_importance(
    model_dict, x_train.columns
)

### SMOTE + PART_NAME

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# CN7, RG3 / LH, RH 구분하는 변수 추가
drop_part['Part']=drop_part['PART_NAME'].str[:3]+drop_part['PART_NAME'].str[-2:]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11',
              'Mold_Temperature_12', 'Mold_Temperature_2','Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7',
              'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL','PART_NAME', 'Reason',
              'Switch_Over_Position', 'TimeStamp', '_id']

drop_col=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
drop_col['PassOrFail']=drop_col['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 더미변수 생성
final_df = pd.get_dummies(data = drop_col, columns = ['Part'], drop_first=True)
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=final_df['PassOrFail'], random_state=0)

# 스케일러
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# SMOTE
smote=SMOTE(sampling_strategy=0.1, k_neighbors=3, random_state=0)
print("Before")
print(x_train_scaled.shape, y_train.shape)

x_train_smote, y_train_smote=smote.fit_resample(x_train_scaled, y_train)
print("After")
print(x_train_smote.shape, y_train_smote.shape)

#### 모델 학습

In [ ]:
# 학습 모델 저장
model_dict={}

# Random Forest
smote_rf=RandomForestClassifier(n_estimators=1000, random_state=0)
smote_rf.fit(x_train_smote, y_train_smote)
model_dict['RandomForest PART_NAME+SMOTE St=0.1']=smote_rf

# Bagging
smote_bag=BaggingClassifier(n_estimators=1000, random_state=0)
smote_bag.fit(x_train_smote, y_train_smote)
model_dict['Bagging PART_NAME+SMOTE St=0.1']=smote_bag

# XGBoost
smote_xgb=XGBClassifier(n_estimators=1000, random_state=0)
smote_xgb.fit(x_train_smote, y_train_smote)
model_dict['XGBoost PART_NAME+SMOTE St=0.1']=smote_xgb

#### 예측

In [ ]:
# model_dict의 키를 모델명으로 그대로 사용해 예측 및 평가
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
# model_dict에 다른 모델이 있어도 Random Forest만 표와 그래프로 출력
importance_tables = show_random_forest_feature_importance(model_dict, x_train.columns)

### SMOTE + Class Weight

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12', 'Mold_Temperature_2',
               'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL',
               'PART_NAME', 'Reason', 'Switch_Over_Position', 'TimeStamp', '_id']

final_df=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
final_df['PassOrFail']=final_df['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 데이터 분할
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=y, random_state=0)

# 스케일링
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# SMOTE Sampling_strategy
smote=SMOTE(sampling_strategy=0.1, k_neighbors=3, random_state=0)
print("Before")
print(x_train_scaled.shape, y_train.shape)

x_train_smote, y_train_smote=smote.fit_resample(x_train_scaled, y_train)
print("After")
print(x_train_smote.shape, y_train_smote.shape)

#### 모델 학습

In [ ]:
model_dict={}
# Random Forest
rf=RandomForestClassifier(n_estimators=1000, class_weight='balanced', min_samples_leaf=2, n_jobs=1,random_state=0)
rf.fit(x_train_smote, y_train_smote)
model_dict['RandomForest SMOTE St=0.1 + Balanced']=rf

#### 예측

In [ ]:
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
importance_tables = show_random_forest_feature_importance(
    model_dict, x_train.columns
)

### SMOTE + PART_NAME + Class Weight

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# CN7, RG3 / LH, RH 구분하는 변수 추가
drop_part['Part']=drop_part['PART_NAME'].str[:3]+drop_part['PART_NAME'].str[-2:]

# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_CD', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11',
              'Mold_Temperature_12', 'Mold_Temperature_2','Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7',
              'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL','PART_NAME', 'Reason',
              'Switch_Over_Position', 'TimeStamp', '_id']

drop_col=drop_part.drop(columns=drop_columns)

# PassOrFail 인코딩
drop_col['PassOrFail']=drop_col['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 더미변수 생성
final_df = pd.get_dummies(data = drop_col, columns = ['Part'], drop_first=True)
x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=final_df['PassOrFail'], random_state=0)

# 스케일러
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# SMOTE
smote=SMOTE(sampling_strategy=0.1, k_neighbors=3, random_state=0)
print("Before")
print(x_train_scaled.shape, y_train.shape)

x_train_smote, y_train_smote=smote.fit_resample(x_train_scaled, y_train)
print("After")
print(x_train_smote.shape, y_train_smote.shape)

#### 모델 학습

In [ ]:
# 학습 모델 저장
model_dict={}

# Random Forest
smote_rf=RandomForestClassifier(n_estimators=1000, class_weight='balanced', min_samples_leaf=2, n_jobs=1,random_state=0)
smote_rf.fit(x_train_smote, y_train_smote)
model_dict['RandomForest PART_NAME+SMOTE St=0.1+Balanced']=smote_rf

In [ ]:
# model_dict의 키를 모델명으로 그대로 사용해 예측 및 평가
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
# model_dict에 다른 모델이 있어도 Random Forest만 표와 그래프로 출력
importance_tables = show_random_forest_feature_importance(model_dict, x_train.columns)

### PART_NAME + SMOTE + 파생변수 추가

#### 데이터 전처리

In [ ]:
# 중복값 제거
drop_dup=labeled.drop_duplicates(keep='first', ignore_index=True)

# CN7, RG3 제외 삭제
drop_part=drop_dup[drop_dup['PART_NAME'].str.startswith(("CN7", "RG3"))]

# CN7, RG3 / LH, RH 구분하는 변수 추가
drop_part['Part']=drop_part['PART_NAME'].str[:3]+drop_part['PART_NAME'].str[-2:]

# TimeStamp 정렬
sort_df=drop_part.copy().sort_values('TimeStamp', ignore_index=True)



# 불필요한 변수 삭제
drop_columns=['Barrel_Temperature_7', 'EQUIP_NAME', 'Mold_Temperature_1', 'Mold_Temperature_10', 'Mold_Temperature_11',
              'Mold_Temperature_12', 'Mold_Temperature_2','Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7',
              'Mold_Temperature_8', 'Mold_Temperature_9', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL','Reason', 'Switch_Over_Position', 'TimeStamp', '_id']

drop_col=sort_df.drop(columns=drop_columns)

# PassOrFail 인코딩
drop_col['PassOrFail']=drop_col['PassOrFail'].map({'Y':0, 'N':1})
# 양품: 0, 불량: 1

# 더미변수 생성
final_df = pd.get_dummies(data = drop_col, columns = ['Part'], drop_first=True)

# 파생변수 추가
final_df['Barrel_Temp_Range']=(final_df[[f'Barrel_Temperature_{i}' for i in range(1, 7)]].max(axis=1)
                                      - final_df[[f'Barrel_Temperature_{i}' for i in range(1, 7)]].min(axis=1))

final_df['Barrel_Temp_Std'] = final_df[[f'Barrel_Temperature_{i}' for i in range(1, 7)]].std(axis=1)

final_df['Mold_Temp_Range']=(final_df[[f'Mold_Temperature_{i}' for i in range(3, 5)]].max(axis=1)
                                      - final_df[[f'Barrel_Temperature_{i}' for i in range(1, 7)]].min(axis=1))

final_df['Mold_Temp_Std'] = final_df[[f'Mold_Temperature_{i}' for i in range(3, 5)]].std(axis=1)

for col in ['Injection_Time', 'Cycle_Time',
            'Max_Injection_Pressure', 'Cushion_Position']:
    final_df[f'{col}_diff'] = final_df[col].diff()
    final_df[f'{col}_rolling_mean_5'] = final_df.groupby(['EQUIP_CD', 'PART_NAME'])[col].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    final_df[f'{col}_rolling_std_5'] = final_df.groupby(['EQUIP_CD', 'PART_NAME'])[col].transform(lambda x: x.shift(1).rolling(5, min_periods=1).std())

drop_columns=['EQUIP_CD', 'PART_NAME']
final_df=final_df.drop(columns=drop_columns)
final_df=final_df.dropna(ignore_index=True)

x=final_df.drop(columns='PassOrFail')
y=final_df['PassOrFail']

# 데이터 분할
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, stratify=y, random_state=0)

# 스케일러
scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# SMOTE
smote=SMOTE(sampling_strategy=0.1, k_neighbors=3, random_state=0)
print("Before")
print(x_train_scaled.shape, y_train.shape)

x_train_smote, y_train_smote=smote.fit_resample(x_train_scaled, y_train)
print("After")
print(x_train_smote.shape, y_train_smote.shape)

#### 모델 학습

In [ ]:
# 학습 모델 저장
model_dict={}

# Random Forest
rf=RandomForestClassifier(n_estimators=1000, random_state=0)
rf.fit(x_train_smote, y_train_smote)
model_dict['RandomForest PART_NAME+SMOTE St=0.1+Feature']=rf

#### 예측

In [ ]:
# model_dict의 키를 모델명으로 그대로 사용해 예측 및 평가
result_df = evaluate_models(model_dict, x_test_scaled, y_test, result_df)
result_df

In [ ]:
# model_dict에 다른 모델이 있어도 Random Forest만 표와 그래프로 출력
importance_tables = show_random_forest_feature_importance(model_dict, x_train.columns)

# 모델링 결과 요약

In [ ]:
result_df.sort_values(['Defect Recall', 'Defect F1 Score', 'Average Precision'], ascending=False)

In [ ]:
result_df.to_csv('04_1_Imbalance_and_Feature_Experiments.csv', index=False)